In [12]:
%pip install numpy pandas seaborn torch torchvision scikit-learn matplotlib graphviz kagglehub "ray[tune]"

Note: you may need to restart the kernel to use updated packages.


In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sajjadtahmasebi/wind-turbine-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/kitton/.cache/kagglehub/datasets/sajjadtahmasebi/wind-turbine-dataset/versions/1


In [14]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sajjadtahmasebi/wind-turbine-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/kitton/.cache/kagglehub/datasets/sajjadtahmasebi/wind-turbine-dataset/versions/1


In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split


df = pd.read_csv(
    "/Users/kitton/.cache/kagglehub/datasets/sajjadtahmasebi/wind-turbine-dataset/versions/1/wind_turbine_dataset.csv"
)

# print(df.info())

# df["turbine_status"].unique().tolist()
# df[df["turbine_status"] == "Off"] = "off"
# le = LabelEncoder()
# df["turbine_status_encoded"] = le.fit_transform(df["turbine_status"])

df = df[pd.to_numeric(df["energy_output"], errors="coerce").notnull()]

df["energy_output"] = df["energy_output"].apply(
    lambda x: 0 if x <= 500 else (1 if x <= 600 else 2)
)

In [16]:
df

,wind_speed,motor_temperature,blade_angle,vibration_level,humidity,air_pressure,energy_output,turbine_status
0,12.490802,56.563558,1.919750,8.618620,28.982391,963.680636,0,needs to be repaired
1,24.014286,22.334208,37.282708,7.893506,74.230604,1057.022743,2,off
2,19.639879,56.735620,11.218841,2.810425,63.106870,995.511060,0,needs to be repaired
3,16.973170,25.380116,12.777181,4.197177,65.321011,1032.785998,0,needs to be repaired
4,8.120373,62.620654,10.181038,5.076866,37.200880,934.721246,0,needs to be repaired
...,...,...,...,...,...,...,...,...
34995,13.688071,55.663792,14.238366,7.877179,53.960014,1073.243533,0,needs to be repaired
34996,14.271533,76.796947,8.328146,1.801420,79.767302,1093.872967,0,needs to be repaired
34997,15.593338,75.255110,26.618267,5.302510,51.949326,1048.203685,0,needs to be repaired
34998,24.351366,32.027862,6.324844,1.031443,57.138669,973.408237,1,optimal


In [17]:
import torch.nn as nn
import torch
import pandas as pd
from sklearn.model_selection import train_test_split

# from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt

In [18]:
df.dropna(inplace=True)  # Drop rows with NaN
X = df.drop(["energy_output", "turbine_status"], axis=1)[:-1].values
y = df["energy_output"][:-1].values

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_valid_tensor = torch.tensor(X_valid, dtype=torch.float32)
y_valid_tensor = torch.tensor(y_valid, dtype=torch.long)

batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_valid_tensor, y_valid_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [19]:
from ray.tune.schedulers import ASHAScheduler
from ray import tune

param_space = {
    "hidden_sizes": tune.grid_search([[128, 64], [256, 128, 64], [128, 128, 64, 32]]),
    "dropout_rates": tune.grid_search(
        [[0.2, 0.2], [0.3, 0.3, 0.2], [0.3, 0.3, 0.2, 0.1]]
    ),
    "activation": tune.grid_search([nn.ReLU, nn.Tanh]),
    "lr": tune.loguniform(1e-5, 1e-2),
    "batch_size": tune.grid_search([16, 32, 64]),
    "optimizer": tune.grid_search(["adam", "sgd"]),
    "epochs": 300,
    "weight_decay": tune.grid_search([0.0, 0.01, 0.001]),
}


scheduler = ASHAScheduler()


In [20]:
import torch
import torch.nn as nn


class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_sizes, activation, dropout_rates):
        super(MLP, self).__init__()
        layers = []
        in_dim = input_dim
        for i, hidden_size in enumerate(hidden_sizes):
            layers.append(nn.Linear(in_dim, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(activation())

            dr = (
                dropout_rates[i]
                if dropout_rates is not None and i < len(dropout_rates)
                else 0.0
            )
            if dr > 0:
                layers.append(nn.Dropout(dr))

            in_dim = hidden_size

        layers.append(nn.Linear(in_dim, output_dim))

        self.layer = nn.Sequential(*layers)

    def forward(self, x):
        return self.layer(x)


In [21]:
import ray
from ray import tune
from ray.air import session

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


def train_model(config):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = MLP(
        input_dim=X_train.shape[1],
        output_dim=3,
        hidden_sizes=config["hidden_sizes"],
        activation=config["activation"],
        dropout_rates=config["dropout_rates"],
    ).to(device)

    if config["optimizer"] == "adam":
        optimizer = optim.Adam(
            model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
        )
    elif config["optimizer"] == "sgd":
        optimizer = optim.SGD(
            model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
        )
    elif config["optimizer"] == "adamw":
        optimizer = optim.AdamW(
            model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
        )

    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        train_dataset, batch_size=config["batch_size"], shuffle=True
    )
    val_loader = DataLoader(
        test_dataset, batch_size=config["batch_size"], shuffle=False
    )

    for epoch in range(config["epochs"]):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb)
                preds = torch.argmax(logits, dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / total

        session.report({"val_accuracy": val_acc})


In [22]:
from ray import tune

analysis = tune.Tuner(
    tune.with_resources(train_model, resources={"cpu": 1}),
    tune_config=tune.TuneConfig(
        metric="val_accuracy",
        mode="max",
        scheduler=scheduler,
    ),
    param_space=param_space,
)


result = analysis.fit()

2025-10-15 14:37:58,797	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'activation': ('__ref_ph', 'bcab2133')}
2025-10-15 14:37:59,749	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'activation': ('__ref_ph', '8cecdf80')}
2025-10-15 14:37:59,802	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'activation': ('__ref_ph', 'bcab2133')}
2025-10-15 14:37:59,826	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'activation': ('__ref_ph', 'bcab2133')}
2025-10-15 14:37:59,868	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'activation': ('__ref_ph', '8cecdf80')}
2025-10-15 14:38:02,658	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'activation': ('__ref_ph', 'bcab2133')}
2025

In [23]:
import pprint

print("🎉[INFO] Training is done!")

best_result = result.get_best_result()

print("\n🎯 Best config:")
pprint.pprint(best_result.config)

print("\n📊 Best result:")
pprint.pprint(best_result)


🎉[INFO] Training is done!

🎯 Best config:
{'activation': <class 'torch.nn.modules.activation.ReLU'>,
 'batch_size': 64,
 'dropout_rates': [0.3, 0.3, 0.2, 0.1],
 'epochs': 300,
 'hidden_sizes': [256, 128, 64],
 'lr': 6.23739868236909e-05,
 'optimizer': 'adam',
 'weight_decay': 0.01}

📊 Best result:
Result(
  metrics={'val_accuracy': 0.8892857142857142},
  path='/Users/kitton/ray_results/train_model_2025-10-15_14-37-51/train_model_daaee_00142_142_activation=ref_ph_8cecdf80,batch_size=64,dropout_rates=0_3_0_3_0_2_0_1,hidden_sizes=256_128_64,lr=0.00_2025-10-15_14-37-53',
  filesystem='local',
  checkpoint=None
)
